# Verifying Fluctuation Theorems in THRML
## Extracting Péclet numbers, Crooks ratios, and entropy production

Any energy-based model sampled by THRML satisfies detailed balance at equilibrium.
This notebook shows how to extract thermodynamic observables from THRML sample
trajectories and verify fluctuation theorem predictions.

**Observables covered:**
1. Drift velocity *v* and diffusion coefficient *D*
2. Péclet number Pe = |v|·L/D (directed vs. diffusive transport)
3. Crooks ratio: P(+ΔE)/P(−ΔE) = exp(ΔE/T)
4. Entropy production rate dS/dt

**References:**
- Crooks, G. E. (1999). Entropy production fluctuation theorem and the nonequilibrium work relation for free energy differences. *Phys. Rev. E*, 60(3), 2721.
- Jarzynski, C. (1997). Nonequilibrium equality for free energies. *Phys. Rev. Lett.*, 78(14), 2690.
- Hack, R. et al. (2022). Crooks fluctuation theorem for general Markov chains.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random
import numpy as np
import matplotlib.pyplot as plt

from thrml.block_management import Block
from thrml.block_sampling import sample_states, SamplingSchedule
from thrml.models.ising import IsingEBM, IsingSamplingProgram
from thrml.pgm import SpinNode

## Build a Simple Biased Ising Chain

We create a 1D Ising chain with N=20 spins. A uniform bias field $h$ breaks symmetry
and creates non-equilibrium directed transport. Two configurations:

- $h = 0.0$ — equilibrium (Pe should be ≈ 0, Crooks ratio ≈ 1)
- $h = 0.5$ — biased (Pe >> 1, Crooks ratio shows exponential asymmetry)

In [ ]:
def build_biased_ising_chain(N=20, h=0.0, J=0.5):
    """Build a 1D Ising chain with uniform bias field h and coupling J.

    Energy: E = -beta * (sum_i h*s_i + sum_<i,j> J*s_i*s_j)
    h > 0 biases spins toward +1 (breaks symmetry).
    """
    nodes = [SpinNode() for _ in range(N)]

    # Uniform bias on each spin
    biases = jnp.full(N, h)

    # Nearest-neighbor coupling (1D chain)
    edges = [(nodes[i], nodes[i + 1]) for i in range(N - 1)]
    weights = jnp.full(len(edges), J)

    beta = jnp.array(1.0)

    ebm = IsingEBM(nodes, edges, biases, weights, beta)
    return ebm, nodes


# Build both configurations
ebm_eq, nodes_eq = build_biased_ising_chain(N=20, h=0.0, J=0.5)
ebm_biased, nodes_biased = build_biased_ising_chain(N=20, h=0.5, J=0.5)

print(f"Equilibrium model: {len(nodes_eq)} spins, h=0.0")
print(f"Biased model: {len(nodes_biased)} spins, h=0.5")

## Sample Trajectories

Run THRML sampling on both configurations and collect per-sweep magnetization trajectories.

In [ ]:
def sample_trajectory(ebm, nodes, n_samples=1000, n_warmup=500, steps_per_sample=5, seed=42):
    """Sample from an Ising model and return magnetization trajectory.

    Returns per-sample mean magnetization (fraction of +1 spins).
    """
    blocks = [Block([node]) for node in nodes]

    program = IsingSamplingProgram(
        ebm=ebm,
        free_blocks=blocks,
        clamped_blocks=[],
    )

    schedule = SamplingSchedule(
        n_warmup=n_warmup,
        n_samples=n_samples,
        steps_per_sample=steps_per_sample,
    )

    init_state = [jnp.array([False]) for _ in nodes]
    key = jax.random.PRNGKey(seed)

    samples = sample_states(
        key=key,
        program=program,
        schedule=schedule,
        init_state_free=init_state,
        state_clamp=[],
        nodes_to_sample=blocks,
    )

    # samples is a list of arrays, one per block, each shape (n_samples, 1)
    all_spins = jnp.stack([s[:, 0] for s in samples], axis=-1)  # (n_samples, N)
    magnetization = jnp.mean(all_spins.astype(jnp.float32), axis=-1)  # (n_samples,)

    return np.array(magnetization)


def compute_energy_trajectory(ebm, nodes, n_samples=1000, n_warmup=500, steps_per_sample=5, seed=42):
    """Sample from an Ising model and return energy trajectory.

    Computes energy from spin configurations: E = -beta * (sum h_i*s_i + sum J_ij*s_i*s_j)
    where s_i in {-1, +1}.
    """
    blocks = [Block([node]) for node in nodes]

    program = IsingSamplingProgram(
        ebm=ebm,
        free_blocks=blocks,
        clamped_blocks=[],
    )

    schedule = SamplingSchedule(
        n_warmup=n_warmup,
        n_samples=n_samples,
        steps_per_sample=steps_per_sample,
    )

    init_state = [jnp.array([False]) for _ in nodes]
    key = jax.random.PRNGKey(seed)

    samples = sample_states(
        key=key,
        program=program,
        schedule=schedule,
        init_state_free=init_state,
        state_clamp=[],
        nodes_to_sample=blocks,
    )

    # Convert boolean samples to spin values {-1, +1}
    all_spins = jnp.stack([s[:, 0] for s in samples], axis=-1)  # (n_samples, N)
    spins = 2.0 * all_spins.astype(jnp.float32) - 1.0  # {False,True} -> {-1,+1}

    # Compute energy per sample
    N = len(nodes)
    h = float(ebm.biases[0])  # uniform bias
    J = float(ebm.weights[0])  # uniform coupling

    bias_energy = -h * jnp.sum(spins, axis=-1)
    coupling_energy = -J * jnp.sum(spins[:, :-1] * spins[:, 1:], axis=-1)
    energy = bias_energy + coupling_energy

    return np.array(energy)


# Sample both configurations
print("Sampling equilibrium system (h=0)...")
traj_eq = sample_trajectory(ebm_eq, nodes_eq, n_samples=1000, seed=42)
energy_eq = compute_energy_trajectory(ebm_eq, nodes_eq, n_samples=1000, seed=42)

print("Sampling biased system (h=0.5)...")
traj_biased = sample_trajectory(ebm_biased, nodes_biased, n_samples=1000, seed=43)
energy_biased = compute_energy_trajectory(ebm_biased, nodes_biased, n_samples=1000, seed=43)

print(f"\nEquilibrium: mean θ = {np.mean(traj_eq):.3f} ± {np.std(traj_eq):.3f}")
print(f"Biased:      mean θ = {np.mean(traj_biased):.3f} ± {np.std(traj_biased):.3f}")

## Extract Péclet Number

The Péclet number diagnoses whether sampling is exploring (diffusion-dominated, Pe < 1)
or being driven (transport-dominated, Pe > 1).

$$ \text{Pe} = \frac{|v| \cdot L}{D} $$

where $v = \langle \Delta\theta / \Delta t \rangle$ and $D = \text{Var}(\Delta\theta / \Delta t) \cdot \Delta t / 2$.

In [ ]:
def compute_peclet(trajectory, L=1.0):
    """Extract Péclet number from a trajectory of observables.

    Pe = |v| * L / D
    where v = mean(Δθ/Δt), D = var(Δθ/Δt) * Δt / 2

    Pe > 1: directed transport dominates
    Pe < 1: diffusion dominates
    Pe ≈ 0: equilibrium (no net drift)
    """
    increments = np.diff(trajectory)
    v = np.mean(increments)
    D = np.var(increments) / 2  # Einstein relation
    if D < 1e-10:
        return 0.0
    return abs(v) * L / D


pe_eq = compute_peclet(traj_eq)
pe_biased = compute_peclet(traj_biased)

print(f"Péclet number (equilibrium, h=0):  Pe = {pe_eq:.4f}")
print(f"Péclet number (biased, h=0.5):     Pe = {pe_biased:.4f}")
print()
print(f"Equilibrium system: {'diffusion-dominated (correct)' if pe_eq < 1 else 'transport-dominated'}")
print(f"Biased system:      {'transport-dominated (correct)' if pe_biased > 1 else 'diffusion-dominated'}")

## Crooks Fluctuation Ratio

The Crooks theorem predicts that the ratio of forward to reverse transition probabilities
follows:

$$ \log \frac{P(+\Delta E)}{P(-\Delta E)} = \frac{\Delta E}{T} $$

On a log-ratio vs. ΔE/T plot, the data should fall on the diagonal (y = x).

In [ ]:
def compute_crooks_ratio(energy_trajectory, T=1.0, n_bins=20):
    """Verify Crooks fluctuation theorem from energy trajectory.

    Returns (bin_centers, log_ratio, theoretical_prediction).
    Theoretical: log(P_fwd/P_rev) = ΔE/T
    """
    dE = np.diff(energy_trajectory)

    # Use symmetric bin edges for fair comparison
    max_abs = np.percentile(np.abs(dE), 95)
    bin_edges = np.linspace(0, max_abs, n_bins + 1)

    hist_pos, _ = np.histogram(dE[dE > 0], bins=bin_edges, density=True)
    hist_neg, _ = np.histogram(-dE[dE < 0], bins=bin_edges, density=True)

    centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Avoid division by zero
    mask = (hist_pos > 0) & (hist_neg > 0)
    log_ratio = np.log(hist_pos[mask] / hist_neg[mask])
    theory = centers[mask] / T

    return centers[mask], log_ratio, theory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: equilibrium
centers_eq, log_ratio_eq, theory_eq = compute_crooks_ratio(energy_eq)
axes[0].scatter(theory_eq, log_ratio_eq, alpha=0.7, s=40, color='steelblue')
lim = max(abs(theory_eq.max()), abs(log_ratio_eq.max()), 1) * 1.2
axes[0].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y = x (theory)')
axes[0].set_xlabel('ΔE / T (theory)')
axes[0].set_ylabel('log(P(+ΔE) / P(−ΔE)) (measured)')
axes[0].set_title('Equilibrium (h = 0)')
axes[0].legend()
axes[0].set_aspect('equal')

# Right: biased
centers_b, log_ratio_b, theory_b = compute_crooks_ratio(energy_biased)
axes[1].scatter(theory_b, log_ratio_b, alpha=0.7, s=40, color='coral')
lim = max(abs(theory_b.max()), abs(log_ratio_b.max()), 1) * 1.2
axes[1].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='y = x (theory)')
axes[1].set_xlabel('ΔE / T (theory)')
axes[1].set_ylabel('log(P(+ΔE) / P(−ΔE)) (measured)')
axes[1].set_title('Biased (h = 0.5)')
axes[1].legend()
axes[1].set_aspect('equal')

plt.suptitle('Crooks Fluctuation Theorem Verification', fontsize=14)
plt.tight_layout()
plt.show()

## Entropy Production Rate

The entropy production rate measures irreversibility per sampling step:

$$ \sigma = \frac{\langle \Delta E \rangle}{T} \text{ per step} $$

Positive $\sigma$: irreversible process (second law). $\sigma \approx 0$: reversible / equilibrium.

In [ ]:
def entropy_production_rate(energy_trajectory, T=1.0):
    """Compute entropy production rate dS/dt from energy changes.

    sigma = <ΔE> / T per step
    Positive sigma: irreversible process (second law)
    sigma ≈ 0: reversible / equilibrium
    """
    dE = np.diff(energy_trajectory)
    sigma = np.mean(dE) / T
    sigma_std = np.std(dE) / T / np.sqrt(len(dE))
    return sigma, sigma_std


sigma_eq, sigma_eq_std = entropy_production_rate(energy_eq)
sigma_biased, sigma_biased_std = entropy_production_rate(energy_biased)

print(f"Entropy production rate (equilibrium):  σ = {sigma_eq:.4f} ± {sigma_eq_std:.4f} per step")
print(f"Entropy production rate (biased):       σ = {sigma_biased:.4f} ± {sigma_biased_std:.4f} per step")

## Summary

In [ ]:
# Crooks R^2 (goodness of fit to y=x line)
def crooks_r2(log_ratio, theory):
    if len(log_ratio) < 2:
        return float('nan')
    ss_res = np.sum((log_ratio - theory) ** 2)
    ss_tot = np.sum((log_ratio - np.mean(log_ratio)) ** 2)
    if ss_tot < 1e-10:
        return float('nan')
    return 1 - ss_res / ss_tot


r2_eq = crooks_r2(log_ratio_eq, theory_eq)
r2_biased = crooks_r2(log_ratio_b, theory_b)

print(f"{'Observable':<25} {'Unbiased (h=0)':>16} {'Biased (h=0.5)':>16}")
print("-" * 60)
print(f"{'Pe':<25} {pe_eq:>16.4f} {pe_biased:>16.4f}")
print(f"{'Crooks R² (y=x fit)':<25} {r2_eq:>16.4f} {r2_biased:>16.4f}")
print(f"{'dS/dt (per step)':<25} {sigma_eq:>16.4f} {sigma_biased:>16.4f}")

## What These Observables Tell You

**Péclet number** diagnoses whether your sampling is exploring (diffusion-dominated,
Pe < 1) or being driven (transport-dominated, Pe > 1). For EBMs used as generative
models, Pe >> 1 during directed sampling is expected. For MCMC equilibrium sampling,
Pe should approach zero after warmup.

**Crooks ratio** verifies that your sampler satisfies detailed balance — the
fundamental requirement for correct Boltzmann sampling. Deviations from the
predicted log-linear relationship indicate implementation issues or insufficient
equilibration.

**Entropy production rate** measures irreversibility per sampling step. This is the
thermodynamic "cost" of non-equilibrium sampling — directly related to the KL
divergence between forward and reverse processes (Jarzynski equality).

These tools apply to any THRML model. Extract the trajectory, compute the observables.

## References

[1] Crooks, G. E. (1999). Entropy production fluctuation theorem. *Phys. Rev. E*, 60(3), 2721.  
[2] Jarzynski, C. (1997). Nonequilibrium equality for free energies. *Phys. Rev. Lett.*, 78(14), 2690.  
[3] Hack, R. et al. (2022). Crooks/Jarzynski for general Markov chains.